# Dates, Categories, and Ranking

Build reliable time-based summaries and ordered categorical reports.

> 🧭 This notebook focuses on practical data work. Examples are intentionally compact and do not cover machine learning.

## Learning goals
- Parse dates and handle invalid date strings.
- Extract year/month/day and use time-based grouping.
- Create ordered categories and ranks.
- Avoid sorting month names alphabetically when calendar order matters.

In [1]:
import pandas as pd

records = pd.DataFrame({
    "date": ["2025-01-04", "2025-02-10", "2025-02-21", "invalid", "2025-03-01"],
    "status": ["High", "Low", "Medium", "High", "Low"],
    "amount": [120, 75, 90, 55, 180],
})
records["date"] = pd.to_datetime(records["date"], errors="coerce")
records["month"] = records["date"].dt.month_name()
records["weekday"] = records["date"].dt.day_name()
display(records)
print("Invalid dates:", records["date"].isna().sum())

Invalid dates: 1


,date,status,amount,month,weekday
0,2025-01-04,High,120,January,Saturday
1,2025-02-10,Low,75,February,Monday
2,2025-02-21,Medium,90,February,Friday
3,NaT,High,55,NaN,NaN
4,2025-03-01,Low,180,March,Saturday


## Monthly summaries and resampling
For daily time series, resampling creates bins such as month-end. It requires a datetime-like index or a datetime column supplied via `on=`.

In [2]:
valid = records.dropna(subset=["date"]).copy()
monthly = valid.set_index("date").resample("ME")["amount"].agg(["sum", "mean", "count"])
display(monthly)

# Group by calendar month number when a full time index is not necessary.
valid["month_num"] = valid["date"].dt.month
display(valid.groupby("month_num", as_index=False)["amount"].sum())

,sum,mean,count
date,,,
2025-01-31,120,120.0,1
2025-02-28,165,82.5,2
2025-03-31,180,180.0,1


,month_num,amount
0,1,120
1,2,165
2,3,180


## Ordered categories and rankings
Categorical order is useful for severity levels, survey responses, and month names. Ranks can be tied using several methods (`average`, `min`, `dense`, `first`).

In [3]:
status_order = ["Low", "Medium", "High"]
records["status"] = pd.Categorical(records["status"], categories=status_order, ordered=True)
display(records.sort_values("status"))

regions = pd.DataFrame({"region": ["North", "South", "East", "West"], "revenue": [200, 200, 150, 95]})
regions["rank_dense"] = regions["revenue"].rank(method="dense", ascending=False).astype(int)
regions["rank_competition"] = regions["revenue"].rank(method="min", ascending=False).astype(int)
display(regions.sort_values(["rank_dense", "region"]))

,date,status,amount,month,weekday
1,2025-02-10,Low,75,February,Monday
4,2025-03-01,Low,180,March,Saturday
2,2025-02-21,Medium,90,February,Friday
0,2025-01-04,High,120,January,Saturday
3,NaT,High,55,NaN,NaN


,region,revenue,rank_dense,rank_competition
0,North,200,1,1
1,South,200,1,1
2,East,150,2,3
3,West,95,3,4


## Practice
Parse the date column, drop only invalid dates for a date-based summary, and rank regions by total revenue.

In [4]:
date_summary = records.dropna(subset=["date"]).groupby(records["date"].dropna().dt.month).agg(rows=("amount", "size"), total=("amount", "sum"))
display(date_summary)
regions["rank"] = regions["revenue"].rank(method="dense", ascending=False).astype(int)
display(regions.sort_values("rank"))

,rows,total
date,,
1,1,120
2,2,165
3,1,180


,region,revenue,rank_dense,rank_competition,rank
0,North,200,1,1,1
1,South,200,1,1,1
2,East,150,2,3,2
3,West,95,3,4,3
